# Iter-SPIN Healthcare — Step 3: generate answers (baseline + suppressed), per snapshot

Generates baseline + suppressed answers for the healthcare fairness (subset-tagged) and privacy eval
questions, for the coupled set(s) at the chosen `ONLY_N` — the inputs to the Step-4 MD-Judge awareness scorer.
**Same generation protocol as the general health Step 3:** greedy, `max_new_tokens=64`, raw questions,
baseline generated (or reused) once, checkpointed per set (resumable). Only the coupled set changes (an iter
snapshot `coupled_iter_health_n{N}.json`, plus `coupled_control_{global,percat}_health_n{N}.json` if present).

Awareness is near-saturated on healthcare (case study §4.4), so run this for **one chosen operating point**
(`ONLY_N`), not the whole ladder. The fairness CSV keeps the `subset` column so Step 4 reports per-subset +
pooled. Suppression is hard zeroing (iter/controls), reversible between sets.

**Inputs (Kaggle Dataset inputs):** `eval_fair_health.csv`, `eval_priv_health.csv`, the coupled snapshot(s)
for `ONLY_N`, and optionally `answers_health_fairness.csv` / `answers_health_privacy.csv` (to reuse the
general-pipeline baseline column instead of regenerating it). Internet ON. GPU **T4×2**. ~3–4 h/set on the
full eval; resumable. **Outputs:** `answers_health_{variant}_n{N}_{fairness,privacy}.csv`,
`results_health_iter_step3.json`.

## Config

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
import subprocess, sys, glob, re, json, time
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.46.3','sentencepiece'], check=False)
import pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID='Qwen/Qwen2-7B-Instruct'
USE_FAST=False
ONLY_N=[1,2]                # iteration budget(s) to generate THIS session (e.g. [10], or [2,5])
ONLY_VARIANTS=['iter','global','percat']   # 'global'/'percat' fire only if their control JSONs are attached
N_EVAL=None                # None = full eval sets; int to subsample (debug)
GEN_BS=4; MAXNEW=64; USE_CHAT_TEMPLATE=False
DTYPE=torch.float16
def find(name):
    for r in ['.','/kaggle/input','/kaggle/working']:
        h=glob.glob(os.path.join(r,'**',name),recursive=True)
        if h: return h[0]
    return None
print('model', MODEL_ID, '| ONLY_N', ONLY_N, '| variants', ONLY_VARIANTS)

## Eval splits + baseline (reuse if attached, else generate)

In [ ]:
EF=find('eval_fair_health.csv'); EP=find('eval_priv_health.csv')
assert EF and EP, 'add eval_fair_health.csv + eval_priv_health.csv (general Step-2 output) as input'
fair_df=pd.read_csv(EF); priv_df=pd.read_csv(EP)
if N_EVAL: fair_df=fair_df.iloc[:N_EVAL].copy(); priv_df=priv_df.iloc[:N_EVAL].copy()
if 'subset' not in fair_df.columns: fair_df['subset']='(all)'
fair_q=fair_df['question'].astype(str).tolist(); fair_sub=fair_df['subset'].astype(str).tolist()
priv_q=priv_df['question'].astype(str).tolist()
print('fairness', len(fair_q), '| privacy', len(priv_q))

def baseline_map(cat, qs):
    # reuse a 'baseline' column from an attached answers_health_*{cat}.csv if present; else {} -> generate
    for pat in [f'answers_health_{cat}.csv', f'answers_health_*{cat}.csv', f'answers_baseline_health_{cat}.csv']:
        f=find(pat)
        if not f: continue
        df=pd.read_csv(f)
        key='q' if 'q' in df.columns else ('question' if 'question' in df.columns else None)
        if 'baseline' in df.columns and key:
            print(f'  baseline {cat} <- {os.path.basename(f)} ({len(df)} rows)')
            return dict(zip(df[key].astype(str), df['baseline'].fillna('').astype(str)))
    print(f'  no reusable baseline for {cat} -> will GENERATE with the unsuppressed model'); return {}
BASE={'fairness':baseline_map('fairness',fair_q),'privacy':baseline_map('privacy',priv_q)}
gaps={c:sum(1 for q in (fair_q if c=='fairness' else priv_q) if str(q) not in BASE[c]) for c in BASE}
print('baseline coverage gaps (0 = fully reused; >0 = generated below):', gaps)

## Model + generation helpers

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=USE_FAST, trust_remote_code=True)
tokenizer.padding_side='left'
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True,
        device_map='auto', max_memory={0:'14GiB',1:'14GiB','cpu':'28GiB'}).eval()
GEN_DEV=model.get_input_embeddings().weight.device
print('model loaded | entry', GEN_DEV)
def fmt(chunk):
    if USE_CHAT_TEMPLATE:
        return [tokenizer.apply_chat_template([{'role':'user','content':q}],tokenize=False,add_generation_prompt=True) for q in chunk]
    return [str(q) for q in chunk]
def suppress_reversible(coupled):
    d=dict(model.named_parameters()); saved=[]; nz=0
    for n,idxlist in coupled.items():
        if n in d and idxlist:
            p=d[n]; idx=torch.tensor(idxlist,device=p.device); flat=p.data.view(-1)
            saved.append((n,idx,flat[idx].clone())); flat[idx]=0.0; nz+=len(idxlist)
    return saved,nz
def restore(saved):
    d=dict(model.named_parameters())
    for n,idx,vals in saved: d[n].data.view(-1)[idx]=vals
@torch.no_grad()
def gen_spin(queries, ckkey, state, CK):
    done=state.get(ckkey,[]); t0=time.time()
    for i in range(len(done),len(queries),GEN_BS):
        enc=tokenizer(fmt(queries[i:i+GEN_BS]),return_tensors='pt',padding=True,truncation=True,max_length=256).to(GEN_DEV)
        g=model.generate(**enc,max_new_tokens=MAXNEW,do_sample=False,pad_token_id=tokenizer.pad_token_id)
        done+=[tokenizer.decode(g[j,enc.input_ids.shape[1]:],skip_special_tokens=True).strip() for j in range(g.shape[0])]
        state[ckkey]=done; json.dump(state,open(CK,'w'))
        print(f'    {ckkey} {len(done)}/{len(queries)} ({time.time()-t0:.0f}s)',end='\r')
    print(); return done
print('gen ready.')

## Discover coupled sets for ONLY_N -> generate -> write answers

In [ ]:
pats={'iter':'coupled_iter_health_n*.json','global':'coupled_control_global_health_n*.json','percat':'coupled_control_percat_health_n*.json'}
SETS=[]
for variant,pat in pats.items():
    if variant not in ONLY_VARIANTS: continue
    seen=set()
    for r in ['.','/kaggle/input','/kaggle/working']:
        for f in glob.glob(os.path.join(r,'**',pat),recursive=True):
            n=int(re.search(r'_n(\d+)\.json$',f).group(1))
            if n in ONLY_N and (variant,n) not in seen:
                seen.add((variant,n)); SETS.append((variant,n,f))
SETS.sort(key=lambda x:(x[1],x[0]))
assert SETS, 'no coupled sets for ONLY_N=%s -- attach coupled_iter_health_n*.json (and controls if any)'%ONLY_N
print('will generate for:', [(v,n) for v,n,_ in SETS])

CK='/kaggle/working/gen_iter_health_ckpt.json'
state=json.load(open(CK)) if os.path.exists(CK) else {}
# baseline (unsuppressed) only where a reused CSV did not fully cover it; BEFORE any suppression
for cat,qs in [('fairness',fair_q),('privacy',priv_q)]:
    if sum(1 for q in qs if str(q) not in BASE[cat])==0: continue
    print(f'[baseline {cat}] generating with unsuppressed model ({len(qs)} q)')
    b=gen_spin(qs, f'baseline_{cat}', state, CK)
    BASE[cat]=dict(zip([str(q) for q in qs], b))
    pd.DataFrame({'q':qs,'baseline':b}).to_csv(f'/kaggle/working/answers_baseline_health_{cat}.csv',index=False)

def changed(a,b): return int(sum(1 for x,y in zip(a,b) if str(x).strip()!=str(y).strip()))
def empties(a): return int(sum(1 for x in a if not str(x).strip()))
res={'model':MODEL_ID,'domain':'healthcare','only_n':ONLY_N,'sets':[]}
for variant,n,f in SETS:
    coupled=json.load(open(f))['coupled']
    saved,nz=suppress_reversible(coupled); print(f'[{variant} n={n}] zeroed {nz}')
    sf=gen_spin(fair_q, f'{variant}_n{n}_fairness', state, CK)
    sp=gen_spin(priv_q, f'{variant}_n{n}_privacy', state, CK)
    restore(saved)
    bf=[BASE['fairness'].get(str(q),'') for q in fair_q]; bp=[BASE['privacy'].get(str(q),'') for q in priv_q]
    pd.DataFrame({'q':fair_q,'subset':fair_sub,'baseline':bf,'spin':sf}).to_csv(f'/kaggle/working/answers_health_{variant}_n{n}_fairness.csv',index=False)
    pd.DataFrame({'q':priv_q,'baseline':bp,'spin':sp}).to_csv(f'/kaggle/working/answers_health_{variant}_n{n}_privacy.csv',index=False)
    r_={'variant':variant,'n':n,'coupled_zeroed':int(nz),'changed_fair':changed(bf,sf),'changed_priv':changed(bp,sp),
        'empty_spin_fair':empties(sf),'empty_spin_priv':empties(sp)}
    res['sets'].append(r_)
    print(f'    wrote answers_health_{variant}_n{n}_*.csv | changed fair {r_["changed_fair"]}/{len(fair_q)} priv {r_["changed_priv"]}/{len(priv_q)} | empties fair {r_["empty_spin_fair"]} priv {r_["empty_spin_priv"]}')
json.dump(res, open('/kaggle/working/results_health_iter_step3.json','w'), indent=2)
print('\nDONE for ONLY_N', ONLY_N, '-> next: Step 4 MD-Judge (per subset + pooled).')